In [1]:
# 1. Setup and imports

import os
import datetime
import numpy as np
from tifffile import imwrite   
import colour                 

# Base output directory (given)
BASE_DIR = "/Users/kate/Documents/retina-model"

# Make dated output folder, e.g. "2025-11-18_bars_gradients"
today_str = datetime.date.today().isoformat()
OUT_DIR = os.path.join(BASE_DIR, today_str + "_bars_gradients")
os.makedirs(OUT_DIR, exist_ok=True)

print("Saving images to:", OUT_DIR)


Saving images to: /Users/kate/Documents/retina-model/2025-11-18_bars_gradients


In [2]:
# 2. CIELAB -> XYZ helper using colour-science (D65 white point)

# CIE 1931 2° D65 white (chromaticity), used by colour-science
ILLUMINANT_D65 = colour.CCS_ILLUMINANTS["CIE 1931 2 Degree Standard Observer"]["D65"]

def lab_to_xyz(L, a, b):
    """
    Thin wrapper around colour.Lab_to_XYZ.
    Input: single L*, a*, b* values.
    Output: X, Y, Z with Y_n = 1.0.
    """
    lab = np.array([L, a, b], dtype=float)
    xyz = colour.Lab_to_XYZ(lab, illuminant=ILLUMINANT_D65)
    X, Y, Z = xyz
    return float(X), float(Y), float(Z)


In [3]:
# 3. Image construction helpers

H, W = 512, 512

def make_uniform_xyz_image(L, a, b):
    """
    Create a 512x512 XYZ image where every pixel has the same Lab color.
    """
    X, Y, Z = lab_to_xyz(L, a, b)
    img = np.zeros((H, W, 3), dtype=np.float32)
    img[..., 0] = X
    img[..., 1] = Y
    img[..., 2] = Z
    return img

def make_contrast_bar_image(bg_lab, stripe_lab, stripe_width=170):
    """
    512x512 image with:
      - background color on two side bars
      - central vertical bar of stripe color

    Widths:
      side bars: 171 pixels each
      stripe:    170 pixels
    """
    bg_L, bg_a, bg_b = bg_lab
    st_L, st_a, st_b = stripe_lab

    # Convert both colors using colour-science
    X_bg, Y_bg, Z_bg = lab_to_xyz(bg_L, bg_a, bg_b)
    X_st, Y_st, Z_st = lab_to_xyz(st_L, st_a, st_b)

    img = np.zeros((H, W, 3), dtype=np.float32)

    side_width = (W - stripe_width) // 2  # 171 for W=512, stripe_width=170
    center_start = side_width
    center_end = side_width + stripe_width

    # Left side background
    img[:, :side_width, 0] = X_bg
    img[:, :side_width, 1] = Y_bg
    img[:, :side_width, 2] = Z_bg

    # Central stripe
    img[:, center_start:center_end, 0] = X_st
    img[:, center_start:center_end, 1] = Y_st
    img[:, center_start:center_end, 2] = Z_st

    # Right side background
    img[:, center_end:, 0] = X_bg
    img[:, center_end:, 1] = Y_bg
    img[:, center_end:, 2] = Z_bg

    return img

def save_xyz(image, filename):
    """
    Save XYZ image as float32 TIFF with naming pattern.
    """
    path = os.path.join(OUT_DIR, filename)
    imwrite(path, image.astype(np.float32))
    print("Saved:", path)


In [4]:
# 4. Contrast bar image sets

# ---- Gray scale image set 1 (contrast bar) ----
# Background (black): (0, 0, 0)
# Stripe L* values: 0, 25, 50, 75, 100

bg_gray1 = (0.0, 0.0, 0.0)
stripe_L_values = [0, 25, 50, 75, 100]

for L in stripe_L_values:
    stripe_lab = (float(L), 0.0, 0.0)
    img = make_contrast_bar_image(bg_gray1, stripe_lab)
    fname = f"bars_gray1_L{L:03d}_XYZ.tiff"
    save_xyz(img, fname)

# ---- Gray scale image set 2 (contrast bar) ----
# Background (gray): (50, 0, 0)

bg_gray2 = (50.0, 0.0, 0.0)

for L in stripe_L_values:
    stripe_lab = (float(L), 0.0, 0.0)
    img = make_contrast_bar_image(bg_gray2, stripe_lab)
    fname = f"bars_gray2_L{L:03d}_XYZ.tiff"
    save_xyz(img, fname)

# ---- Blue-Yellow scale image set 1 (contrast bar) ----
# Background (blue): (50, 0, -100)
# Stripe b* values: -100, -50, 0, 50, 100

bg_by1 = (50.0, 0.0, -100.0)
stripe_b_values = [-100, -50, 0, 50, 100]

for b in stripe_b_values:
    stripe_lab = (50.0, 0.0, float(b))
    img = make_contrast_bar_image(bg_by1, stripe_lab)
    fname = f"bars_by1_b{b:+04d}_XYZ.tiff"  
    save_xyz(img, fname)

# ---- Red-Green scale image set 1 (contrast bar) ----
# Background (red): (50, 100, 0)
# Stripe a* values: -100, -50, 0, 50, 100

bg_rg1 = (50.0, 100.0, 0.0)
stripe_a_values = [-100, -50, 0, 50, 100]

for a in stripe_a_values:
    stripe_lab = (50.0, float(a), 0.0)
    img = make_contrast_bar_image(bg_rg1, stripe_lab)
    fname = f"bars_rg1_a{a:+04d}_XYZ.tiff"  
    save_xyz(img, fname)


Saved: /Users/kate/Documents/retina-model/2025-11-18_bars_gradients/bars_gray1_L000_XYZ.tiff
Saved: /Users/kate/Documents/retina-model/2025-11-18_bars_gradients/bars_gray1_L025_XYZ.tiff
Saved: /Users/kate/Documents/retina-model/2025-11-18_bars_gradients/bars_gray1_L050_XYZ.tiff
Saved: /Users/kate/Documents/retina-model/2025-11-18_bars_gradients/bars_gray1_L075_XYZ.tiff
Saved: /Users/kate/Documents/retina-model/2025-11-18_bars_gradients/bars_gray1_L100_XYZ.tiff
Saved: /Users/kate/Documents/retina-model/2025-11-18_bars_gradients/bars_gray2_L000_XYZ.tiff
Saved: /Users/kate/Documents/retina-model/2025-11-18_bars_gradients/bars_gray2_L025_XYZ.tiff
Saved: /Users/kate/Documents/retina-model/2025-11-18_bars_gradients/bars_gray2_L050_XYZ.tiff
Saved: /Users/kate/Documents/retina-model/2025-11-18_bars_gradients/bars_gray2_L075_XYZ.tiff
Saved: /Users/kate/Documents/retina-model/2025-11-18_bars_gradients/bars_gray2_L100_XYZ.tiff
Saved: /Users/kate/Documents/retina-model/2025-11-18_bars_gradients/ba

In [5]:
# 5. Gradient image sets

def make_gray_gradient_image():
    """
    Gray scale image set 3:
      Left half: white (L* = 100, a* = 0, b* = 0)
      Right half: L* gradient from 0 to 100, a* = 0, b* = 0
    """
    img = np.zeros((H, W, 3), dtype=np.float32)

    # Left half (white)
    X_w, Y_w, Z_w = lab_to_xyz(100.0, 0.0, 0.0)
    img[:, :W//2, 0] = X_w
    img[:, :W//2, 1] = Y_w
    img[:, :W//2, 2] = Z_w

    # Right half: L* gradient from 0 -> 100
    right_width = W // 2  # 256
    for i in range(right_width):
        t = i / (right_width - 1)  # 0 -> 1
        L = 0.0 * (1 - t) + 100.0 * t
        X, Y, Z = lab_to_xyz(L, 0.0, 0.0)
        col = W//2 + i
        img[:, col, 0] = X
        img[:, col, 1] = Y
        img[:, col, 2] = Z

    return img

def make_by_gradient_image():
    """
    Blue-Yellow scale image set 2:
      Left half: white (100, 0, 0)
      Right half: b* gradient from -100 (blue) to +100 (yellow),
                  L* = 50, a* = 0
    """
    img = np.zeros((H, W, 3), dtype=np.float32)

    # Left half: white
    X_w, Y_w, Z_w = lab_to_xyz(100.0, 0.0, 0.0)
    img[:, :W//2, 0] = X_w
    img[:, :W//2, 1] = Y_w
    img[:, :W//2, 2] = Z_w

    # Right half: b* gradient
    right_width = W // 2
    for i in range(right_width):
        t = i / (right_width - 1)  # 0 -> 1
        b = -100.0 * (1 - t) + 100.0 * t  # -100 -> +100
        X, Y, Z = lab_to_xyz(50.0, 0.0, b)
        col = W//2 + i
        img[:, col, 0] = X
        img[:, col, 1] = Y
        img[:, col, 2] = Z

    return img

def make_rg_gradient_image():
    """
    Red-Green scale image set 2:
      Left half: white (100, 0, 0)
      Right half: a* gradient from -100 (green) to +100 (red),
                  L* = 50, b* = 0
    """
    img = np.zeros((H, W, 3), dtype=np.float32)

    # Left half: white
    X_w, Y_w, Z_w = lab_to_xyz(100.0, 0.0, 0.0)
    img[:, :W//2, 0] = X_w
    img[:, :W//2, 1] = Y_w
    img[:, :W//2, 2] = Z_w

    # Right half: a* gradient
    right_width = W // 2
    for i in range(right_width):
        t = i / (right_width - 1)  # 0 -> 1
        a = -100.0 * (1 - t) + 100.0 * t  # -100 -> +100
        X, Y, Z = lab_to_xyz(50.0, a, 0.0)
        col = W//2 + i
        img[:, col, 0] = X
        img[:, col, 1] = Y
        img[:, col, 2] = Z

    return img

# Generate and save the three gradient images
img_gray_grad = make_gray_gradient_image()
save_xyz(img_gray_grad, "gradient_gray3_XYZ.tiff")

img_by_grad = make_by_gradient_image()
save_xyz(img_by_grad, "gradient_by2_XYZ.tiff")

img_rg_grad = make_rg_gradient_image()
save_xyz(img_rg_grad, "gradient_rg2_XYZ.tiff")


Saved: /Users/kate/Documents/retina-model/2025-11-18_bars_gradients/gradient_gray3_XYZ.tiff
Saved: /Users/kate/Documents/retina-model/2025-11-18_bars_gradients/gradient_by2_XYZ.tiff
Saved: /Users/kate/Documents/retina-model/2025-11-18_bars_gradients/gradient_rg2_XYZ.tiff
